# ADS v2: event stream

Unions the 334 event families into `ads_v2.EventStream` (one row per EventId: the same event from a repeated source row is kept once) and checks it.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("19_EventStream", "started")


In [ ]:
/******************************************************************************************************/
/*  Name        : ads_v2.EventStream                                                                    */
/*  Description : Every event family, one row per EventId.                                              */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_evt_a0, stg.adsv2_evt_a0_f1, stg.adsv2_evt_a3_e1, stg.adsv2_evt_a4a, */
/*              :  stg.adsv2_evt_a4b, stg.adsv2_evt_a4_f1, stg.adsv2_evt_a5a, stg.adsv2_evt_a5_f1, stg. */
/*              : adsv2_evt_a1, stg.adsv2_evt_a1_e1, stg.adsv2_evt_a2, stg.adsv2_evt_a2_f1, stg.adsv2_e */
/*              : vt_a3, stg.adsv2_evt_a3_f1, stg.adsv2_evt_a3_b1_1, stg.adsv2_evt_a2_e1, stg.adsv2_evt */
/*              : _a5_1, stg.adsv2_evt_a6_b1_1, stg.adsv2_evt_a6, stg.adsv2_evt_a6_b1_4, stg.adsv2_evt_ */
/*              : a6_f1, stg.adsv2_evt_a6_b1_f1, stg.adsv2_evt_a7_f1, stg.adsv2_evt_a7_b1_4, stg.adsv2_ */
/*              : evt_a6_b1_2, stg.adsv2_evt_a6_b1_3, stg.adsv2_evt_a6_b1_6, stg.adsv2_evt_a7, stg.adsv */
/*              : 2_evt_a6_b1_5, stg.adsv2_evt_a6_b1_o1, stg.adsv2_evt_a7_b1, stg.adsv2_evt_a7_b1_1, st */
/*              : g.adsv2_evt_a7_b1_2, stg.adsv2_evt_a7_b1_3, stg.adsv2_evt_a7_b1_5, stg.adsv2_evt_r0,  */
/*              : stg.adsv2_evt_r0_f1, stg.adsv2_evt_r0_b1, stg.adsv2_evt_r0_f2, stg.adsv2_evt_r1a, stg */
/*              : .adsv2_evt_r1b, stg.adsv2_evt_r1_b2, stg.adsv2_evt_r1_e1, stg.adsv2_evt_r1_e1_f3, stg */
/*              : .adsv2_evt_r2a, stg.adsv2_evt_r2b, stg.adsv2_evt_r2_f1, stg.adsv2_evt_r2_b1_f1, stg.a */
/*              : dsv2_evt_r2_e1, stg.adsv2_evt_r3a, stg.adsv2_evt_r3b, stg.adsv2_evt_r3c, stg.adsv2_ev */
/*              : t_r3d, stg.adsv2_evt_r3b_f1, stg.adsv2_evt_r3_o1, stg.adsv2_evt_r4, stg.adsv2_evt_r4_ */
/*              : f1, stg.adsv2_evt_r4a, stg.adsv2_evt_r4c, stg.adsv2_evt_r5, stg.adsv2_evt_r6, stg.ads */
/*              : v2_evt_r2_b1_1, stg.adsv2_evt_r2_b1_2, stg.adsv2_evt_r2_b1_3, stg.adsv2_evt_r2_b1_3_f */
/*              : 1, stg.adsv2_evt_r2_b1_4_f1, stg.adsv2_evt_r2_b1_o1, stg.adsv2_evt_r2_b1_o2, stg.adsv */
/*              : 2_evt_r2_b1_o3, stg.adsv2_evt_r3_b1, stg.adsv2_evt_r3_b1_1, stg.adsv2_evt_r3_b1_1_f1, */
/*              :  stg.adsv2_evt_r3_b1_2, stg.adsv2_evt_r3_b1_2_f1, stg.adsv2_evt_r3_b1_3, stg.adsv2_ev */
/*              : t_r3_b1_4, stg.adsv2_evt_r3_b1_o1, stg.adsv2_evt_r3_b1_o3, stg.adsv2_evt_r6_f1, stg.a */
/*              : dsv2_evt_r4_b2_f1, stg.adsv2_evt_r4_b2_1, stg.adsv2_evt_r4_b2_2, stg.adsv2_evt_r4_b2_ */
/*              : 4, stg.adsv2_evt_r4_b2_5, stg.adsv2_evt_r4_b2_3, stg.adsv2_evt_r4_b2_o1, stg.adsv2_ev */
/*              : t_r4_b2_6, stg.adsv2_evt_r6_b1, stg.adsv2_evt_r6_b1_1, stg.adsv2_evt_r6_b1_2, stg.ads */
/*              : v2_evt_r6_b1_5, stg.adsv2_evt_ha1, stg.adsv2_evt_ha2, stg.adsv2_evt_ha2_f1, stg.adsv2 */
/*              : _evt_ha3, stg.adsv2_evt_ha3_f1, stg.adsv2_evt_ha0, stg.adsv2_evt_ha0_f1, stg.adsv2_ev */
/*              : t_ha1_e2, stg.adsv2_evt_ha1_e1, stg.adsv2_evt_ha2_e1, stg.adsv2_evt_ha1_f1, stg.adsv2 */
/*              : _evt_ha4a, stg.adsv2_evt_ha4b, stg.adsv2_evt_ha4_f1, stg.adsv2_evt_ha5_f1, stg.adsv2_ */
/*              : evt_ha6_b1_1, stg.adsv2_evt_ha6, stg.adsv2_evt_ha6_b1_4, stg.adsv2_evt_ha6_f1, stg.ad */
/*              : sv2_evt_ha7_b1_4, stg.adsv2_evt_ha6_b1_5, stg.adsv2_evt_ha6_b1_2, stg.adsv2_evt_ha6_b */
/*              : 1_3, stg.adsv2_evt_ha6_b1_6, stg.adsv2_evt_ha5_1, stg.adsv2_evt_ha7, stg.adsv2_evt_ha */
/*              : 7_b1_1, stg.adsv2_evt_ha6_b1_o1, stg.adsv2_evt_hr0, stg.adsv2_evt_hr0_f1, stg.adsv2_e */
/*              : vt_hr0_f2, stg.adsv2_evt_hr0_f3, stg.adsv2_evt_hr1a, stg.adsv2_evt_hr1b, stg.adsv2_ev */
/*              : t_hr1_e1, stg.adsv2_evt_hr1_e1_f1, stg.adsv2_evt_hr2a, stg.adsv2_evt_hr2b, stg.adsv2_ */
/*              : evt_hr2_f1, stg.adsv2_evt_hr3b_f1, stg.adsv2_evt_hr3a, stg.adsv2_evt_hr4a, stg.adsv2_ */
/*              : evt_hr4c, stg.adsv2_evt_hr3b, stg.adsv2_evt_hr3_o1, stg.adsv2_evt_hr4, stg.adsv2_evt_ */
/*              : hr3_b1_1, stg.adsv2_evt_hr2_b1_1, stg.adsv2_evt_hr2_b1_f1, stg.adsv2_evt_hr2_b1_2, st */
/*              : g.adsv2_evt_hr2_b1_2_f1, stg.adsv2_evt_hr2_b1_3_f1, stg.adsv2_evt_hr2_b1_o1, stg.adsv */
/*              : 2_evt_hr2_b1_o2, stg.adsv2_evt_hr2_b1_o3, stg.adsv2_evt_hr3_b1_1_f1, stg.adsv2_evt_hr */
/*              : 3_b1_2, stg.adsv2_evt_hr3_b1_2_f1, stg.adsv2_evt_hr3_b1_o1, stg.adsv2_evt_hr3_b1_o3,  */
/*              : stg.adsv2_evt_hr5, stg.adsv2_evt_hr6, stg.adsv2_evt_hr6_f1, stg.adsv2_evt_hr4_b2_f1,  */
/*              : stg.adsv2_evt_hr4_b2_4, stg.adsv2_evt_hr4_b2_1, stg.adsv2_evt_hr4_b2_2, stg.adsv2_evt */
/*              : _hr4_b2_5, stg.adsv2_evt_hr4_b2_3, stg.adsv2_evt_hr4_b2_6, stg.adsv2_evt_hr4_b2_o1, s */
/*              : tg.adsv2_evt_hr6_b1_1, stg.adsv2_evt_hr6_b1_2, stg.adsv2_evt_hr6_b1_5, stg.adsv2_evt_ */
/*              : m1, stg.adsv2_evt_m1_f1, stg.adsv2_evt_m_v1, stg.adsv2_evt_m_v2, stg.adsv2_evt_m_v3,  */
/*              : stg.adsv2_evt_m_v4, stg.adsv2_evt_m3, stg.adsv2_evt_m4, stg.adsv2_evt_m4a, stg.adsv2_ */
/*              : evt_m6, stg.adsv2_evt_m3_b1_1, stg.adsv2_evt_m3_b1_2, stg.adsv2_evt_m3_b1_4, stg.adsv */
/*              : 2_evt_m3_b1_5, stg.adsv2_evt_m3_b1_3, stg.adsv2_evt_m3_b1_6, stg.adsv2_evt_c1a, stg.a */
/*              : dsv2_evt_c1b, stg.adsv2_evt_c1a_f1, stg.adsv2_evt_c2, stg.adsv2_evt_c3, stg.adsv2_evt */
/*              : _c4, stg.adsv2_evt_c5, stg.adsv2_evt_c6, stg.adsv2_evt_arr_1, stg.adsv2_evt_arr_2, st */
/*              : g.adsv2_evt_arr_o1, stg.adsv2_evt_arr_3, stg.adsv2_evt_arr_o2, stg.adsv2_evt_cl1a, st */
/*              : g.adsv2_evt_cl1b, stg.adsv2_evt_cl1c, stg.adsv2_evt_cl3_f1, stg.adsv2_evt_cl4, stg.ad */
/*              : sv2_evt_cl_o1, stg.adsv2_evt_cl_o2, stg.adsv2_evt_d1, stg.adsv2_evt_d1_f1, stg.adsv2_ */
/*              : evt_d3_f1, stg.adsv2_evt_d_v2, stg.adsv2_evt_d_v3, stg.adsv2_evt_d_v4, stg.adsv2_evt_ */
/*              : d_v5, stg.adsv2_evt_d_v6, stg.adsv2_evt_hm1, stg.adsv2_evt_hm1_f1, stg.adsv2_evt_hm2_ */
/*              : o1, stg.adsv2_evt_hm3, stg.adsv2_evt_hm4, stg.adsv2_evt_hm4a, stg.adsv2_evt_hm4c, stg */
/*              : .adsv2_evt_hm6, stg.adsv2_evt_hm6_b1_1, stg.adsv2_evt_hm6_b1_2, stg.adsv2_evt_hm4_b1_ */
/*              : f1, stg.adsv2_evt_hm4_b1_1, stg.adsv2_evt_hm4_b1_4, stg.adsv2_evt_hm4_b1_5, stg.adsv2 */
/*              : _evt_hm4_b1_2, stg.adsv2_evt_hm4_b1_3, stg.adsv2_evt_hm4_b1_6, stg.adsv2_evt_hm4_b1_o */
/*              : 1, stg.adsv2_evt_hc1a, stg.adsv2_evt_hc1a_f1, stg.adsv2_evt_hc2, stg.adsv2_evt_hc3, s */
/*              : tg.adsv2_evt_hc4, stg.adsv2_evt_hc5, stg.adsv2_evt_hc6, stg.adsv2_evt_harr_1, stg.ads */
/*              : v2_evt_harr_2, stg.adsv2_evt_harr_o1, stg.adsv2_evt_harr_3, stg.adsv2_evt_harr_o2, st */
/*              : g.adsv2_evt_hcl1a, stg.adsv2_evt_hcl1b, stg.adsv2_evt_hcl1c, stg.adsv2_evt_hcl_o2, st */
/*              : g.adsv2_evt_hd1, stg.adsv2_evt_hd1_f1, stg.adsv2_evt_hd3_f1, stg.adsv2_evt_vr1a, stg. */
/*              : adsv2_evt_vr1b, stg.adsv2_evt_vr1_e1, stg.adsv2_evt_vr2a, stg.adsv2_evt_vr2b, stg.ads */
/*              : v2_evt_vr3b_f1, stg.adsv2_evt_vr3a, stg.adsv2_evt_vr3b, stg.adsv2_evt_vr3_o1, stg.ads */
/*              : v2_evt_vr4, stg.adsv2_evt_vr5, stg.adsv2_evt_vr3_b1_1, stg.adsv2_evt_vr3_b1_1_f1, stg */
/*              : .adsv2_evt_vr3_b1_2, stg.adsv2_evt_vr3_b1_2_f1, stg.adsv2_evt_vr3_b1_o1, stg.adsv2_ev */
/*              : t_vr3_b1_o3, stg.adsv2_evt_vr4_b2, stg.adsv2_evt_vr4_b2_f1, stg.adsv2_evt_vr4_b2_4, s */
/*              : tg.adsv2_evt_vr4_b2_1, stg.adsv2_evt_vr4_b2_2, stg.adsv2_evt_vr4_b2_5, stg.adsv2_evt_ */
/*              : vr4_b2_3, stg.adsv2_evt_vr4_b2_6, stg.adsv2_evt_vr4_b2_o1, stg.adsv2_evt_vr6_b1_1, st */
/*              : g.adsv2_evt_va1, stg.adsv2_evt_va0, stg.adsv2_evt_va0_f1, stg.adsv2_evt_va5, stg.adsv */
/*              : 2_evt_va5_b1_4, stg.adsv2_evt_va6, stg.adsv2_evt_va6_b1_f1, stg.adsv2_evt_va6_b1_1, s */
/*              : tg.adsv2_evt_va6_b1_4, stg.adsv2_evt_va6_b1_2, stg.adsv2_evt_va6_b1_3, stg.adsv2_evt_ */
/*              : va6_b1_5, stg.adsv2_evt_va6_b1_6, stg.adsv2_evt_va6_b1_o1, stg.adsv2_evt_va7, stg.ads */
/*              : v2_evt_va7_b1_1, stg.adsv2_evt_va7_b1_2, stg.adsv2_evt_va7_b1_3, stg.adsv2_evt_va7_b1 */
/*              : _5, stg.adsv2_evt_vm1, stg.adsv2_evt_vm1_f1, stg.adsv2_evt_vm3, stg.adsv2_evt_vm4c, s */
/*              : tg.adsv2_evt_vm2_o1, stg.adsv2_evt_vm6, stg.adsv2_evt_vm6_b1_1, stg.adsv2_evt_vm6_b1_ */
/*              : 2, stg.adsv2_evt_vm3_b1_1, stg.adsv2_evt_vm3_b1_2, stg.adsv2_evt_vm3_b1_3, stg.adsv2_ */
/*              : evt_vm3_b1_6, stg.adsv2_evt_vm3_b1_4, stg.adsv2_evt_vm3_b1_5, stg.adsv2_evt_vm3_b1_f1 */
/*              : , stg.adsv2_evt_vm3_b1_o1, stg.adsv2_evt_vc1a, stg.adsv2_evt_vc2, stg.adsv2_evt_vc3,  */
/*              : stg.adsv2_evt_vc4, stg.adsv2_evt_vc5, stg.adsv2_evt_vc6, stg.adsv2_evt_varr_1, stg.ad */
/*              : sv2_evt_varr_2, stg.adsv2_evt_varr_3, stg.adsv2_evt_varr_4, stg.adsv2_evt_varr_o1, st */
/*              : g.adsv2_evt_varr_o2, stg.adsv2_evt_vcl1a, stg.adsv2_evt_vcl1b, stg.adsv2_evt_vcl1c, s */
/*              : tg.adsv2_evt_vd1, stg.adsv2_evt_vd1_f1, stg.adsv2_evt_vd3_f1, stg.adsv2_evt_vd_v1, st */
/*              : g.adsv2_evt_ta1, stg.adsv2_evt_ta1p, stg.adsv2_evt_ta5, stg.adsv2_evt_ta3_b1_2, stg.a */
/*              : dsv2_evt_tr1_e1, stg.adsv2_evt_tr4, stg.adsv2_evt_tr_o1                               */
/******************************************************************************************************/
create or replace table ${ads.target}.EventStream as
select  EventId, EventTypeId, EventDescription, PolicyTypeGroup, Grain, EventDateTime, EventDate, EventTimeBasis,
        SourceSystemId, EventSourceReference, SourcePolicyReference, SourceQuoteReference, SourceCustomerReference, SourceCustomerReferenceType, PolicyId, QuoteId,
        AdsBatchId, current_timestamp() as AdsLoadDateTime
from    (
        select  *, row_number() over (partition by EventId order by EventDateTime) as rn
        from    (
                select * from ${ads.stg}.adsv2_evt_a0
                union all
                select * from ${ads.stg}.adsv2_evt_a0_f1
                union all
                select * from ${ads.stg}.adsv2_evt_a3_e1
                union all
                select * from ${ads.stg}.adsv2_evt_a4a
                union all
                select * from ${ads.stg}.adsv2_evt_a4b
                union all
                select * from ${ads.stg}.adsv2_evt_a4_f1
                union all
                select * from ${ads.stg}.adsv2_evt_a5a
                union all
                select * from ${ads.stg}.adsv2_evt_a5_f1
                union all
                select * from ${ads.stg}.adsv2_evt_a1
                union all
                select * from ${ads.stg}.adsv2_evt_a1_e1
                union all
                select * from ${ads.stg}.adsv2_evt_a2
                union all
                select * from ${ads.stg}.adsv2_evt_a2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_a3
                union all
                select * from ${ads.stg}.adsv2_evt_a3_f1
                union all
                select * from ${ads.stg}.adsv2_evt_a3_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_a2_e1
                union all
                select * from ${ads.stg}.adsv2_evt_a5_1
                union all
                select * from ${ads.stg}.adsv2_evt_a6_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_a6
                union all
                select * from ${ads.stg}.adsv2_evt_a6_b1_4
                union all
                select * from ${ads.stg}.adsv2_evt_a6_f1
                union all
                select * from ${ads.stg}.adsv2_evt_a6_b1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_a7_f1
                union all
                select * from ${ads.stg}.adsv2_evt_a7_b1_4
                union all
                select * from ${ads.stg}.adsv2_evt_a6_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_a6_b1_3
                union all
                select * from ${ads.stg}.adsv2_evt_a6_b1_6
                union all
                select * from ${ads.stg}.adsv2_evt_a7
                union all
                select * from ${ads.stg}.adsv2_evt_a6_b1_5
                union all
                select * from ${ads.stg}.adsv2_evt_a6_b1_o1
                union all
                select * from ${ads.stg}.adsv2_evt_a7_b1
                union all
                select * from ${ads.stg}.adsv2_evt_a7_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_a7_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_a7_b1_3
                union all
                select * from ${ads.stg}.adsv2_evt_a7_b1_5
                union all
                select * from ${ads.stg}.adsv2_evt_r0
                union all
                select * from ${ads.stg}.adsv2_evt_r0_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r0_b1
                union all
                select * from ${ads.stg}.adsv2_evt_r0_f2
                union all
                select * from ${ads.stg}.adsv2_evt_r1a
                union all
                select * from ${ads.stg}.adsv2_evt_r1b
                union all
                select * from ${ads.stg}.adsv2_evt_r1_b2
                union all
                select * from ${ads.stg}.adsv2_evt_r1_e1
                union all
                select * from ${ads.stg}.adsv2_evt_r1_e1_f3
                union all
                select * from ${ads.stg}.adsv2_evt_r2a
                union all
                select * from ${ads.stg}.adsv2_evt_r2b
                union all
                select * from ${ads.stg}.adsv2_evt_r2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r2_b1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r2_e1
                union all
                select * from ${ads.stg}.adsv2_evt_r3a
                union all
                select * from ${ads.stg}.adsv2_evt_r3b
                union all
                select * from ${ads.stg}.adsv2_evt_r3c
                union all
                select * from ${ads.stg}.adsv2_evt_r3d
                union all
                select * from ${ads.stg}.adsv2_evt_r3b_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r3_o1
                union all
                select * from ${ads.stg}.adsv2_evt_r4
                union all
                select * from ${ads.stg}.adsv2_evt_r4_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r4a
                union all
                select * from ${ads.stg}.adsv2_evt_r4c
                union all
                select * from ${ads.stg}.adsv2_evt_r5
                union all
                select * from ${ads.stg}.adsv2_evt_r6
                union all
                select * from ${ads.stg}.adsv2_evt_r2_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_r2_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_r2_b1_3
                union all
                select * from ${ads.stg}.adsv2_evt_r2_b1_3_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r2_b1_4_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r2_b1_o1
                union all
                select * from ${ads.stg}.adsv2_evt_r2_b1_o2
                union all
                select * from ${ads.stg}.adsv2_evt_r2_b1_o3
                union all
                select * from ${ads.stg}.adsv2_evt_r3_b1
                union all
                select * from ${ads.stg}.adsv2_evt_r3_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_r3_b1_1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r3_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_r3_b1_2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r3_b1_3
                union all
                select * from ${ads.stg}.adsv2_evt_r3_b1_4
                union all
                select * from ${ads.stg}.adsv2_evt_r3_b1_o1
                union all
                select * from ${ads.stg}.adsv2_evt_r3_b1_o3
                union all
                select * from ${ads.stg}.adsv2_evt_r6_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r4_b2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_r4_b2_1
                union all
                select * from ${ads.stg}.adsv2_evt_r4_b2_2
                union all
                select * from ${ads.stg}.adsv2_evt_r4_b2_4
                union all
                select * from ${ads.stg}.adsv2_evt_r4_b2_5
                union all
                select * from ${ads.stg}.adsv2_evt_r4_b2_3
                union all
                select * from ${ads.stg}.adsv2_evt_r4_b2_o1
                union all
                select * from ${ads.stg}.adsv2_evt_r4_b2_6
                union all
                select * from ${ads.stg}.adsv2_evt_r6_b1
                union all
                select * from ${ads.stg}.adsv2_evt_r6_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_r6_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_r6_b1_5
                union all
                select * from ${ads.stg}.adsv2_evt_ha1
                union all
                select * from ${ads.stg}.adsv2_evt_ha2
                union all
                select * from ${ads.stg}.adsv2_evt_ha2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_ha3
                union all
                select * from ${ads.stg}.adsv2_evt_ha3_f1
                union all
                select * from ${ads.stg}.adsv2_evt_ha0
                union all
                select * from ${ads.stg}.adsv2_evt_ha0_f1
                union all
                select * from ${ads.stg}.adsv2_evt_ha1_e2
                union all
                select * from ${ads.stg}.adsv2_evt_ha1_e1
                union all
                select * from ${ads.stg}.adsv2_evt_ha2_e1
                union all
                select * from ${ads.stg}.adsv2_evt_ha1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_ha4a
                union all
                select * from ${ads.stg}.adsv2_evt_ha4b
                union all
                select * from ${ads.stg}.adsv2_evt_ha4_f1
                union all
                select * from ${ads.stg}.adsv2_evt_ha5_f1
                union all
                select * from ${ads.stg}.adsv2_evt_ha6_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_ha6
                union all
                select * from ${ads.stg}.adsv2_evt_ha6_b1_4
                union all
                select * from ${ads.stg}.adsv2_evt_ha6_f1
                union all
                select * from ${ads.stg}.adsv2_evt_ha7_b1_4
                union all
                select * from ${ads.stg}.adsv2_evt_ha6_b1_5
                union all
                select * from ${ads.stg}.adsv2_evt_ha6_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_ha6_b1_3
                union all
                select * from ${ads.stg}.adsv2_evt_ha6_b1_6
                union all
                select * from ${ads.stg}.adsv2_evt_ha5_1
                union all
                select * from ${ads.stg}.adsv2_evt_ha7
                union all
                select * from ${ads.stg}.adsv2_evt_ha7_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_ha6_b1_o1
                union all
                select * from ${ads.stg}.adsv2_evt_hr0
                union all
                select * from ${ads.stg}.adsv2_evt_hr0_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr0_f2
                union all
                select * from ${ads.stg}.adsv2_evt_hr0_f3
                union all
                select * from ${ads.stg}.adsv2_evt_hr1a
                union all
                select * from ${ads.stg}.adsv2_evt_hr1b
                union all
                select * from ${ads.stg}.adsv2_evt_hr1_e1
                union all
                select * from ${ads.stg}.adsv2_evt_hr1_e1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr2a
                union all
                select * from ${ads.stg}.adsv2_evt_hr2b
                union all
                select * from ${ads.stg}.adsv2_evt_hr2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr3b_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr3a
                union all
                select * from ${ads.stg}.adsv2_evt_hr4a
                union all
                select * from ${ads.stg}.adsv2_evt_hr4c
                union all
                select * from ${ads.stg}.adsv2_evt_hr3b
                union all
                select * from ${ads.stg}.adsv2_evt_hr3_o1
                union all
                select * from ${ads.stg}.adsv2_evt_hr4
                union all
                select * from ${ads.stg}.adsv2_evt_hr3_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_hr2_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_hr2_b1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr2_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_hr2_b1_2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr2_b1_3_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr2_b1_o1
                union all
                select * from ${ads.stg}.adsv2_evt_hr2_b1_o2
                union all
                select * from ${ads.stg}.adsv2_evt_hr2_b1_o3
                union all
                select * from ${ads.stg}.adsv2_evt_hr3_b1_1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr3_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_hr3_b1_2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr3_b1_o1
                union all
                select * from ${ads.stg}.adsv2_evt_hr3_b1_o3
                union all
                select * from ${ads.stg}.adsv2_evt_hr5
                union all
                select * from ${ads.stg}.adsv2_evt_hr6
                union all
                select * from ${ads.stg}.adsv2_evt_hr6_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr4_b2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hr4_b2_4
                union all
                select * from ${ads.stg}.adsv2_evt_hr4_b2_1
                union all
                select * from ${ads.stg}.adsv2_evt_hr4_b2_2
                union all
                select * from ${ads.stg}.adsv2_evt_hr4_b2_5
                union all
                select * from ${ads.stg}.adsv2_evt_hr4_b2_3
                union all
                select * from ${ads.stg}.adsv2_evt_hr4_b2_6
                union all
                select * from ${ads.stg}.adsv2_evt_hr4_b2_o1
                union all
                select * from ${ads.stg}.adsv2_evt_hr6_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_hr6_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_hr6_b1_5
                union all
                select * from ${ads.stg}.adsv2_evt_m1
                union all
                select * from ${ads.stg}.adsv2_evt_m1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_m_v1
                union all
                select * from ${ads.stg}.adsv2_evt_m_v2
                union all
                select * from ${ads.stg}.adsv2_evt_m_v3
                union all
                select * from ${ads.stg}.adsv2_evt_m_v4
                union all
                select * from ${ads.stg}.adsv2_evt_m3
                union all
                select * from ${ads.stg}.adsv2_evt_m4
                union all
                select * from ${ads.stg}.adsv2_evt_m4a
                union all
                select * from ${ads.stg}.adsv2_evt_m6
                union all
                select * from ${ads.stg}.adsv2_evt_m3_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_m3_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_m3_b1_4
                union all
                select * from ${ads.stg}.adsv2_evt_m3_b1_5
                union all
                select * from ${ads.stg}.adsv2_evt_m3_b1_3
                union all
                select * from ${ads.stg}.adsv2_evt_m3_b1_6
                union all
                select * from ${ads.stg}.adsv2_evt_c1a
                union all
                select * from ${ads.stg}.adsv2_evt_c1b
                union all
                select * from ${ads.stg}.adsv2_evt_c1a_f1
                union all
                select * from ${ads.stg}.adsv2_evt_c2
                union all
                select * from ${ads.stg}.adsv2_evt_c3
                union all
                select * from ${ads.stg}.adsv2_evt_c4
                union all
                select * from ${ads.stg}.adsv2_evt_c5
                union all
                select * from ${ads.stg}.adsv2_evt_c6
                union all
                select * from ${ads.stg}.adsv2_evt_arr_1
                union all
                select * from ${ads.stg}.adsv2_evt_arr_2
                union all
                select * from ${ads.stg}.adsv2_evt_arr_o1
                union all
                select * from ${ads.stg}.adsv2_evt_arr_3
                union all
                select * from ${ads.stg}.adsv2_evt_arr_o2
                union all
                select * from ${ads.stg}.adsv2_evt_cl1a
                union all
                select * from ${ads.stg}.adsv2_evt_cl1b
                union all
                select * from ${ads.stg}.adsv2_evt_cl1c
                union all
                select * from ${ads.stg}.adsv2_evt_cl3_f1
                union all
                select * from ${ads.stg}.adsv2_evt_cl4
                union all
                select * from ${ads.stg}.adsv2_evt_cl_o1
                union all
                select * from ${ads.stg}.adsv2_evt_cl_o2
                union all
                select * from ${ads.stg}.adsv2_evt_d1
                union all
                select * from ${ads.stg}.adsv2_evt_d1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_d3_f1
                union all
                select * from ${ads.stg}.adsv2_evt_d_v2
                union all
                select * from ${ads.stg}.adsv2_evt_d_v3
                union all
                select * from ${ads.stg}.adsv2_evt_d_v4
                union all
                select * from ${ads.stg}.adsv2_evt_d_v5
                union all
                select * from ${ads.stg}.adsv2_evt_d_v6
                union all
                select * from ${ads.stg}.adsv2_evt_hm1
                union all
                select * from ${ads.stg}.adsv2_evt_hm1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hm2_o1
                union all
                select * from ${ads.stg}.adsv2_evt_hm3
                union all
                select * from ${ads.stg}.adsv2_evt_hm4
                union all
                select * from ${ads.stg}.adsv2_evt_hm4a
                union all
                select * from ${ads.stg}.adsv2_evt_hm4c
                union all
                select * from ${ads.stg}.adsv2_evt_hm6
                union all
                select * from ${ads.stg}.adsv2_evt_hm6_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_hm6_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_hm4_b1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hm4_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_hm4_b1_4
                union all
                select * from ${ads.stg}.adsv2_evt_hm4_b1_5
                union all
                select * from ${ads.stg}.adsv2_evt_hm4_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_hm4_b1_3
                union all
                select * from ${ads.stg}.adsv2_evt_hm4_b1_6
                union all
                select * from ${ads.stg}.adsv2_evt_hm4_b1_o1
                union all
                select * from ${ads.stg}.adsv2_evt_hc1a
                union all
                select * from ${ads.stg}.adsv2_evt_hc1a_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hc2
                union all
                select * from ${ads.stg}.adsv2_evt_hc3
                union all
                select * from ${ads.stg}.adsv2_evt_hc4
                union all
                select * from ${ads.stg}.adsv2_evt_hc5
                union all
                select * from ${ads.stg}.adsv2_evt_hc6
                union all
                select * from ${ads.stg}.adsv2_evt_harr_1
                union all
                select * from ${ads.stg}.adsv2_evt_harr_2
                union all
                select * from ${ads.stg}.adsv2_evt_harr_o1
                union all
                select * from ${ads.stg}.adsv2_evt_harr_3
                union all
                select * from ${ads.stg}.adsv2_evt_harr_o2
                union all
                select * from ${ads.stg}.adsv2_evt_hcl1a
                union all
                select * from ${ads.stg}.adsv2_evt_hcl1b
                union all
                select * from ${ads.stg}.adsv2_evt_hcl1c
                union all
                select * from ${ads.stg}.adsv2_evt_hcl_o2
                union all
                select * from ${ads.stg}.adsv2_evt_hd1
                union all
                select * from ${ads.stg}.adsv2_evt_hd1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_hd3_f1
                union all
                select * from ${ads.stg}.adsv2_evt_vr1a
                union all
                select * from ${ads.stg}.adsv2_evt_vr1b
                union all
                select * from ${ads.stg}.adsv2_evt_vr1_e1
                union all
                select * from ${ads.stg}.adsv2_evt_vr2a
                union all
                select * from ${ads.stg}.adsv2_evt_vr2b
                union all
                select * from ${ads.stg}.adsv2_evt_vr3b_f1
                union all
                select * from ${ads.stg}.adsv2_evt_vr3a
                union all
                select * from ${ads.stg}.adsv2_evt_vr3b
                union all
                select * from ${ads.stg}.adsv2_evt_vr3_o1
                union all
                select * from ${ads.stg}.adsv2_evt_vr4
                union all
                select * from ${ads.stg}.adsv2_evt_vr5
                union all
                select * from ${ads.stg}.adsv2_evt_vr3_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_vr3_b1_1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_vr3_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_vr3_b1_2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_vr3_b1_o1
                union all
                select * from ${ads.stg}.adsv2_evt_vr3_b1_o3
                union all
                select * from ${ads.stg}.adsv2_evt_vr4_b2
                union all
                select * from ${ads.stg}.adsv2_evt_vr4_b2_f1
                union all
                select * from ${ads.stg}.adsv2_evt_vr4_b2_4
                union all
                select * from ${ads.stg}.adsv2_evt_vr4_b2_1
                union all
                select * from ${ads.stg}.adsv2_evt_vr4_b2_2
                union all
                select * from ${ads.stg}.adsv2_evt_vr4_b2_5
                union all
                select * from ${ads.stg}.adsv2_evt_vr4_b2_3
                union all
                select * from ${ads.stg}.adsv2_evt_vr4_b2_6
                union all
                select * from ${ads.stg}.adsv2_evt_vr4_b2_o1
                union all
                select * from ${ads.stg}.adsv2_evt_vr6_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_va1
                union all
                select * from ${ads.stg}.adsv2_evt_va0
                union all
                select * from ${ads.stg}.adsv2_evt_va0_f1
                union all
                select * from ${ads.stg}.adsv2_evt_va5
                union all
                select * from ${ads.stg}.adsv2_evt_va5_b1_4
                union all
                select * from ${ads.stg}.adsv2_evt_va6
                union all
                select * from ${ads.stg}.adsv2_evt_va6_b1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_va6_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_va6_b1_4
                union all
                select * from ${ads.stg}.adsv2_evt_va6_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_va6_b1_3
                union all
                select * from ${ads.stg}.adsv2_evt_va6_b1_5
                union all
                select * from ${ads.stg}.adsv2_evt_va6_b1_6
                union all
                select * from ${ads.stg}.adsv2_evt_va6_b1_o1
                union all
                select * from ${ads.stg}.adsv2_evt_va7
                union all
                select * from ${ads.stg}.adsv2_evt_va7_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_va7_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_va7_b1_3
                union all
                select * from ${ads.stg}.adsv2_evt_va7_b1_5
                union all
                select * from ${ads.stg}.adsv2_evt_vm1
                union all
                select * from ${ads.stg}.adsv2_evt_vm1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_vm3
                union all
                select * from ${ads.stg}.adsv2_evt_vm4c
                union all
                select * from ${ads.stg}.adsv2_evt_vm2_o1
                union all
                select * from ${ads.stg}.adsv2_evt_vm6
                union all
                select * from ${ads.stg}.adsv2_evt_vm6_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_vm6_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_vm3_b1_1
                union all
                select * from ${ads.stg}.adsv2_evt_vm3_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_vm3_b1_3
                union all
                select * from ${ads.stg}.adsv2_evt_vm3_b1_6
                union all
                select * from ${ads.stg}.adsv2_evt_vm3_b1_4
                union all
                select * from ${ads.stg}.adsv2_evt_vm3_b1_5
                union all
                select * from ${ads.stg}.adsv2_evt_vm3_b1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_vm3_b1_o1
                union all
                select * from ${ads.stg}.adsv2_evt_vc1a
                union all
                select * from ${ads.stg}.adsv2_evt_vc2
                union all
                select * from ${ads.stg}.adsv2_evt_vc3
                union all
                select * from ${ads.stg}.adsv2_evt_vc4
                union all
                select * from ${ads.stg}.adsv2_evt_vc5
                union all
                select * from ${ads.stg}.adsv2_evt_vc6
                union all
                select * from ${ads.stg}.adsv2_evt_varr_1
                union all
                select * from ${ads.stg}.adsv2_evt_varr_2
                union all
                select * from ${ads.stg}.adsv2_evt_varr_3
                union all
                select * from ${ads.stg}.adsv2_evt_varr_4
                union all
                select * from ${ads.stg}.adsv2_evt_varr_o1
                union all
                select * from ${ads.stg}.adsv2_evt_varr_o2
                union all
                select * from ${ads.stg}.adsv2_evt_vcl1a
                union all
                select * from ${ads.stg}.adsv2_evt_vcl1b
                union all
                select * from ${ads.stg}.adsv2_evt_vcl1c
                union all
                select * from ${ads.stg}.adsv2_evt_vd1
                union all
                select * from ${ads.stg}.adsv2_evt_vd1_f1
                union all
                select * from ${ads.stg}.adsv2_evt_vd3_f1
                union all
                select * from ${ads.stg}.adsv2_evt_vd_v1
                union all
                select * from ${ads.stg}.adsv2_evt_ta1
                union all
                select * from ${ads.stg}.adsv2_evt_ta1p
                union all
                select * from ${ads.stg}.adsv2_evt_ta5
                union all
                select * from ${ads.stg}.adsv2_evt_ta3_b1_2
                union all
                select * from ${ads.stg}.adsv2_evt_tr1_e1
                union all
                select * from ${ads.stg}.adsv2_evt_tr4
                union all
                select * from ${ads.stg}.adsv2_evt_tr_o1
                ) u
        ) d
where   rn = 1

## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to AdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "EventStream", "bk": ["EventId"], "sk": null, "nullable": [], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "AdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.AdsBuildLog")
if failures:
    ads_run_log("19_EventStream", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ads checks failed:\n- " + "\n- ".join(failures))
ads_run_log("19_EventStream", "succeeded")


## Output tests per family

The habits list (bugs and lessons, 16 October): every family's row count, first and last time, events before the window start or after its end, and families with no rows or the same count and times as another family (identical steps). Reported in `ads_v2.AdsFamilyCheck`, not failed: each flag is a question to answer in writing.


In [ ]:
%%pyspark
# Output tests per family: count, first / last time, events outside the window, zero-row and identical families.
from pyspark.sql import functions as F
codes = ["A0", "A0.F1", "A3.E1", "A4a", "A4b", "A4.F1", "A5a", "A5.F1", "A1", "A1.E1", "A2", "A2.F1", "A3", "A3.F1", "A3.B1.1", "A2.E1", "A5.1", "A6.B1.1", "A6", "A6.B1.4", "A6.F1", "A6.B1.F1", "A7.F1", "A7.B1.4", "A6.B1.2", "A6.B1.3", "A6.B1.6", "A7", "A6.B1.5", "A6.B1.O1", "A7.B1", "A7.B1.1", "A7.B1.2", "A7.B1.3", "A7.B1.5", "R0", "R0.F1", "R0.B1", "R0.F2", "R1a", "R1b", "R1.B2", "R1.E1", "R1.E1.F3", "R2a", "R2b", "R2.F1", "R2.B1.F1", "R2.E1", "R3a", "R3b", "R3c", "R3d", "R3b.F1", "R3.O1", "R4", "R4.F1", "R4a", "R4c", "R5", "R6", "R2.B1.1", "R2.B1.2", "R2.B1.3", "R2.B1.3.F1", "R2.B1.4.F1", "R2.B1.O1", "R2.B1.O2", "R2.B1.O3", "R3.B1", "R3.B1.1", "R3.B1.1.F1", "R3.B1.2", "R3.B1.2.F1", "R3.B1.3", "R3.B1.4", "R3.B1.O1", "R3.B1.O3", "R6.F1", "R4.B2.F1", "R4.B2.1", "R4.B2.2", "R4.B2.4", "R4.B2.5", "R4.B2.3", "R4.B2.O1", "R4.B2.6", "R6.B1", "R6.B1.1", "R6.B1.2", "R6.B1.5", "HA1", "HA2", "HA2.F1", "HA3", "HA3.F1", "HA0", "HA0.F1", "HA1.E2", "HA1.E1", "HA2.E1", "HA1.F1", "HA4a", "HA4b", "HA4.F1", "HA5.F1", "HA6.B1.1", "HA6", "HA6.B1.4", "HA6.F1", "HA7.B1.4", "HA6.B1.5", "HA6.B1.2", "HA6.B1.3", "HA6.B1.6", "HA5.1", "HA7", "HA7.B1.1", "HA6.B1.O1", "HR0", "HR0.F1", "HR0.F2", "HR0.F3", "HR1a", "HR1b", "HR1.E1", "HR1.E1.F1", "HR2a", "HR2b", "HR2.F1", "HR3b.F1", "HR3a", "HR4a", "HR4c", "HR3b", "HR3.O1", "HR4", "HR3.B1.1", "HR2.B1.1", "HR2.B1.F1", "HR2.B1.2", "HR2.B1.2.F1", "HR2.B1.3.F1", "HR2.B1.O1", "HR2.B1.O2", "HR2.B1.O3", "HR3.B1.1.F1", "HR3.B1.2", "HR3.B1.2.F1", "HR3.B1.O1", "HR3.B1.O3", "HR5", "HR6", "HR6.F1", "HR4.B2.F1", "HR4.B2.4", "HR4.B2.1", "HR4.B2.2", "HR4.B2.5", "HR4.B2.3", "HR4.B2.6", "HR4.B2.O1", "HR6.B1.1", "HR6.B1.2", "HR6.B1.5", "M1", "M1.F1", "M.V1", "M.V2", "M.V3", "M.V4", "M3", "M4", "M4a", "M6", "M3.B1.1", "M3.B1.2", "M3.B1.4", "M3.B1.5", "M3.B1.3", "M3.B1.6", "C1a", "C1b", "C1a.F1", "C2", "C3", "C4", "C5", "C6", "ARR.1", "ARR.2", "ARR.O1", "ARR.3", "ARR.O2", "CL1a", "CL1b", "CL1c", "CL3.F1", "CL4", "CL.O1", "CL.O2", "D1", "D1.F1", "D3.F1", "D.V2", "D.V3", "D.V4", "D.V5", "D.V6", "HM1", "HM1.F1", "HM2.O1", "HM3", "HM4", "HM4a", "HM4c", "HM6", "HM6.B1.1", "HM6.B1.2", "HM4.B1.F1", "HM4.B1.1", "HM4.B1.4", "HM4.B1.5", "HM4.B1.2", "HM4.B1.3", "HM4.B1.6", "HM4.B1.O1", "HC1a", "HC1a.F1", "HC2", "HC3", "HC4", "HC5", "HC6", "HARR.1", "HARR.2", "HARR.O1", "HARR.3", "HARR.O2", "HCL1a", "HCL1b", "HCL1c", "HCL.O2", "HD1", "HD1.F1", "HD3.F1", "VR1a", "VR1b", "VR1.E1", "VR2a", "VR2b", "VR3b.F1", "VR3a", "VR3b", "VR3.O1", "VR4", "VR5", "VR3.B1.1", "VR3.B1.1.F1", "VR3.B1.2", "VR3.B1.2.F1", "VR3.B1.O1", "VR3.B1.O3", "VR4.B2", "VR4.B2.F1", "VR4.B2.4", "VR4.B2.1", "VR4.B2.2", "VR4.B2.5", "VR4.B2.3", "VR4.B2.6", "VR4.B2.O1", "VR6.B1.1", "VA1", "VA0", "VA0.F1", "VA5", "VA5.B1.4", "VA6", "VA6.B1.F1", "VA6.B1.1", "VA6.B1.4", "VA6.B1.2", "VA6.B1.3", "VA6.B1.5", "VA6.B1.6", "VA6.B1.O1", "VA7", "VA7.B1.1", "VA7.B1.2", "VA7.B1.3", "VA7.B1.5", "VM1", "VM1.F1", "VM3", "VM4c", "VM2.O1", "VM6", "VM6.B1.1", "VM6.B1.2", "VM3.B1.1", "VM3.B1.2", "VM3.B1.3", "VM3.B1.6", "VM3.B1.4", "VM3.B1.5", "VM3.B1.F1", "VM3.B1.O1", "VC1a", "VC2", "VC3", "VC4", "VC5", "VC6", "VARR.1", "VARR.2", "VARR.3", "VARR.4", "VARR.O1", "VARR.O2", "VCL1a", "VCL1b", "VCL1c", "VD1", "VD1.F1", "VD3.F1", "VD.V1", "TA1", "TA1p", "TA5", "TA3.B1.2", "TR1.E1", "TR4", "TR.O1"]
es = spark.table(f"{target_schema}.EventStream")
st = (es.groupBy("EventTypeId")
        .agg(F.count("*").alias("Events"), F.min("EventDateTime").alias("FirstEvent"), F.max("EventDateTime").alias("LastEvent"),
             F.sum(F.when(F.col("EventDateTime") < F.lit(start_time).cast("timestamp"), 1).otherwise(0)).alias("BeforeWindow"),
             F.sum(F.when(F.col("EventDateTime") >= F.lit(end_time).cast("timestamp"), 1).otherwise(0)).alias("AfterWindow"),
             F.sum(F.when(F.col("EventTimeBasis") == "derived", 1).otherwise(0)).alias("DerivedTimes"))
        .toPandas())
import pandas as pd
st = pd.DataFrame({"EventTypeId": codes}).merge(st, on="EventTypeId", how="left").fillna({"Events": 0})
sig = st[st.Events > 0].groupby(["Events", "FirstEvent", "LastEvent"])["EventTypeId"].transform(lambda s: ", ".join(sorted(s)) if len(s) > 1 else "")
st["SameAs"] = sig.reindex(st.index).fillna("")
st["Flags"] = [", ".join(f for f, c in [("no events", r.Events == 0), ("before window", (r.BeforeWindow or 0) > 0),
                                         ("after window", (r.AfterWindow or 0) > 0), ("identical to " + r.SameAs, bool(r.SameAs))] if c)
               for r in st.itertuples()]
st["AdsBatchId"] = batch_id
(spark.createDataFrame(st.astype({"Events": "int64"}).where(pd.notnull(st), None))
      .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{target_schema}.AdsFamilyCheck"))
flagged = st[st.Flags != ""]
print(f"{len(st)} families, {len(flagged)} flagged")
print(flagged[["EventTypeId", "Events", "BeforeWindow", "AfterWindow", "Flags"]].to_string(index=False))
